# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 256.38it/s]


2026-04-21 10:17:00.890 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:907 - Data batch-empirical estimation of propensity score.


2026-04-21 10:17:00.897 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:956 - Data prediction of expected reward based on gbm model.


2026-04-21 10:17:02.287 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1073 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-04-21 10:17:32.240 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-04-21 10:17:32.643 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-04-21 10:17:33.044 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.505373,0.472186,0.538480,0.016947,b-ipw,reward_0
1,0.468630,0.467715,0.469597,0.000484,dm,reward_0
2,0.505756,0.473209,0.537793,0.016509,dr,reward_0
3,0.468630,0.467675,0.469574,0.000482,dros-opt,reward_0
4,0.505756,0.473479,0.537818,0.016412,dros-pess,reward_0
5,0.506051,0.473687,0.539882,0.016865,ipw,reward_0
6,0.505705,0.472905,0.539380,0.016957,rep,reward_0
7,0.505731,0.473607,0.537844,0.016538,sndr,reward_0
8,0.505719,0.472961,0.540303,0.017169,snips,reward_0
9,0.505756,0.472905,0.538807,0.016673,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 293.87it/s]


2026-04-21 10:17:33.605 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1305 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<08:11,  2.03it/s]

...

2026-04-21 10:18:17.699 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-04-21 10:18:18.101 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-04-21 10:18:18.503 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.514341,0.477091,0.554037,0.019555,b-ipw,reward_0
1,0.469307,0.468322,0.470332,0.000509,dm,reward_0
2,0.509336,0.470623,0.546904,0.019490,dr,reward_0
3,0.469307,0.468312,0.470310,0.000510,dros-opt,reward_0
4,0.509336,0.471752,0.547193,0.019269,dros-pess,reward_0
5,0.514445,0.472117,0.560247,0.022456,ipw,reward_0
6,0.509131,0.467887,0.555480,0.022173,rep,reward_0
7,0.508923,0.472138,0.546401,0.019089,sndr,reward_0
8,0.509139,0.466235,0.553320,0.022267,snips,reward_0
9,0.509336,0.471075,0.547708,0.019454,sg-dr,reward_0
